In [1]:

# PART 3 - FILE I/O, APIs & EXCEPTION HANDLING
# Theme: Product Explorer & Error-Resilient Logger

import requests
from datetime import datetime

# HELPER FUNCTION - ERROR LOGGER

def log_error(function_name, error_type, message):
    with open("error_log.txt", "a", encoding="utf-8") as file:
        timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        file.write(f"[{timestamp}] ERROR in {function_name}: {error_type} — {message}\n")

# TASK 1 - FILE READ & WRITE BASICS

print("========== TASK 1 - FILE READ & WRITE BASICS ==========\n")

# Part A - Write
notes = [
    "Topic 1: Variables store data. Python is dynamically typed.\n",
    "Topic 2: Lists are ordered and mutable.\n",
    "Topic 3: Dictionaries store key-value pairs.\n",
    "Topic 4: Loops automate repetitive tasks.\n",
    "Topic 5: Exception handling prevents crashes.\n"
]

with open("python_notes.txt", "w", encoding="utf-8") as file:
    file.writelines(notes)

print("File written successfully.")

with open("python_notes.txt", "a", encoding="utf-8") as file:
    file.write("Topic 6: Functions help organize code.\n")
    file.write("Topic 7: APIs allow programs to communicate.\n")

print("Lines appended.\n")

# Part B - Read
try:
    with open("python_notes.txt", "r", encoding="utf-8") as file:
        lines = file.readlines()

    print("Reading file contents:")
    for i, line in enumerate(lines, start=1):
        print(f"{i}. {line.strip()}")

    print(f"\nTotal number of lines: {len(lines)}\n")

    # Keyword search
    keyword = "python"   # you can change this if needed
    print(f'Searching for keyword: "{keyword}"')
    found = False

    for line in lines:
        if keyword.lower() in line.lower():
            print(line.strip())
            found = True

    if not found:
        print("No matching lines found.")

except Exception as e:
    print("Error reading file:", e)
    log_error("task1_file_read", type(e).__name__, str(e))

# TASK 2 - API INTEGRATION

print("\n========== TASK 2 - API INTEGRATION ==========\n")

base_url = "https://dummyjson.com/products"

# Step 1 - Fetch and Display Products
def fetch_products():
    try:
        response = requests.get(f"{base_url}?limit=20", timeout=5)
        response.raise_for_status()
        data = response.json()
        products = data["products"]

        print("ID | Title | Category | Price | Rating")
        print("-" * 70)
        for product in products:
            print(f'{product["id"]} | {product["title"]} | {product["category"]} | ${product["price"]} | {product["rating"]}')

        return products

    except requests.exceptions.ConnectionError:
        print("Connection failed. Please check your internet.")
        log_error("fetch_products", "ConnectionError", "No internet connection")
    except requests.exceptions.Timeout:
        print("Request timed out. Try again later.")
        log_error("fetch_products", "Timeout", "Request timed out")
    except Exception as e:
        print("Unexpected error:", e)
        log_error("fetch_products", type(e).__name__, str(e))
        return []

products = fetch_products()

# Step 2 - Filter and Sort
print("\nFiltered Products (Rating >= 4.5, sorted by price descending):")
filtered_products = [p for p in products if p["rating"] >= 4.5]
filtered_products.sort(key=lambda x: x["price"], reverse=True)

for product in filtered_products:
    print(f'{product["title"]} | ${product["price"]} | Rating: {product["rating"]}')

# Step 3 - Search by Category
print("\nLaptops Category Products:")
try:
    response = requests.get(f"{base_url}/category/laptops", timeout=5)
    response.raise_for_status()
    laptop_data = response.json()

    for product in laptop_data["products"]:
        print(f'{product["title"]} - ${product["price"]}')

except requests.exceptions.ConnectionError:
    print("Connection failed. Please check your internet.")
    log_error("fetch_laptops", "ConnectionError", "No internet connection")
except requests.exceptions.Timeout:
    print("Request timed out. Try again later.")
    log_error("fetch_laptops", "Timeout", "Request timed out")
except Exception as e:
    print("Unexpected error:", e)
    log_error("fetch_laptops", type(e).__name__, str(e))

# Step 4 - POST Request
print("\nPOST Request (Simulated Create Product):")
new_product = {
    "title": "My Custom Product",
    "price": 999,
    "category": "electronics",
    "description": "A product I created via API"
}

try:
    response = requests.post(f"{base_url}/add", json=new_product, timeout=5)
    response.raise_for_status()
    print(response.json())

except requests.exceptions.ConnectionError:
    print("Connection failed. Please check your internet.")
    log_error("post_product", "ConnectionError", "No internet connection")
except requests.exceptions.Timeout:
    print("Request timed out. Try again later.")
    log_error("post_product", "Timeout", "Request timed out")
except Exception as e:
    print("Unexpected error:", e)
    log_error("post_product", type(e).__name__, str(e))

# TASK 3 - EXCEPTION HANDLING

print("\n========== TASK 3 - EXCEPTION HANDLING ==========\n")

# Part A - Guarded Calculator
def safe_divide(a, b):
    try:
        return a / b
    except ZeroDivisionError:
        return "Error: Cannot divide by zero"
    except TypeError:
        return "Error: Invalid input types"

print("safe_divide(10, 2):", safe_divide(10, 2))
print("safe_divide(10, 0):", safe_divide(10, 0))
print('safe_divide("ten", 2):', safe_divide("ten", 2))

# Part B - Guarded File Reader
def read_file_safe(filename):
    try:
        with open(filename, "r", encoding="utf-8") as file:
            return file.read()
    except FileNotFoundError:
        print(f"Error: File '{filename}' not found.")
        log_error("read_file_safe", "FileNotFoundError", f"{filename} not found")
    finally:
        print("File operation attempt complete.")

print("\nReading python_notes.txt:")
content = read_file_safe("python_notes.txt")
if content:
    print(content)

print("Reading ghost_file.txt:")
read_file_safe("ghost_file.txt")

# Part C - Robust API Calls
print("\nRobust API Call Test:")
try:
    response = requests.get(f"{base_url}?limit=5", timeout=5)
    response.raise_for_status()
    print("API call successful.")
except requests.exceptions.ConnectionError:
    print("Connection failed. Please check your internet.")
    log_error("robust_api_call", "ConnectionError", "No internet connection")
except requests.exceptions.Timeout:
    print("Request timed out. Try again later.")
    log_error("robust_api_call", "Timeout", "Request timed out")
except Exception as e:
    print("Unexpected error:", e)
    log_error("robust_api_call", type(e).__name__, str(e))

# Part D - Input Validation Loop
print("\nProduct Lookup Demo:")

test_inputs = ["5", "999", "abc", "101", "quit"]   # auto demo inputs for Colab

for user_input in test_inputs:
    print(f"Input: {user_input}")

    if user_input.lower() == "quit":
        print("Exiting product lookup.")
        break

    if not user_input.isdigit():
        print("Warning: Please enter a valid integer.\n")
        continue

    product_id = int(user_input)

    if product_id < 1 or product_id > 100:
        print("Warning: Product ID must be between 1 and 100.\n")
        continue

    try:
        response = requests.get(f"{base_url}/{product_id}", timeout=5)

        if response.status_code == 404:
            print("Product not found.")
            log_error("lookup_product", "HTTPError", f"404 Not Found for product ID {product_id}")
        elif response.status_code == 200:
            product = response.json()
            print(f'Product Found: {product["title"]} - ${product["price"]}')
        else:
            print("Unexpected response received.")
            log_error("lookup_product", "HTTPError", f"Unexpected status code {response.status_code}")

    except requests.exceptions.ConnectionError:
        print("Connection failed. Please check your internet.")
        log_error("lookup_product", "ConnectionError", "No internet connection")
    except requests.exceptions.Timeout:
        print("Request timed out. Try again later.")
        log_error("lookup_product", "Timeout", "Request timed out")
    except Exception as e:
        print("Unexpected error:", e)
        log_error("lookup_product", type(e).__name__, str(e))

    print()

# TASK 4 - LOGGING TO FILE

print("\n========== TASK 4 - LOGGING TO FILE ==========\n")

# Intentionally trigger ConnectionError
try:
    requests.get("https://this-host-does-not-exist-xyz.com/api", timeout=5)
except requests.exceptions.ConnectionError as e:
    print("Intentional ConnectionError triggered.")
    log_error("intentional_connection_test", "ConnectionError", str(e))

# Intentionally trigger HTTP-like error using invalid product ID
try:
    response = requests.get(f"{base_url}/999", timeout=5)
    if response.status_code != 200:
        print("Intentional product lookup error triggered.")
        log_error("intentional_http_test", "HTTPError", "404 Not Found for product ID 999")
except Exception as e:
    log_error("intentional_http_test", type(e).__name__, str(e))

# Read and print error log
print("Contents of error_log.txt:\n")
try:
    with open("error_log.txt", "r", encoding="utf-8") as file:
        print(file.read())
except Exception as e:
    print("Could not read error log:", e)


========== TASK 1 - FILE READ & WRITE BASICS ==========

File written successfully.
Lines appended.

Reading file contents:
1. Topic 1: Variables store data. Python is dynamically typed.
2. Topic 2: Lists are ordered and mutable.
3. Topic 3: Dictionaries store key-value pairs.
4. Topic 4: Loops automate repetitive tasks.
5. Topic 5: Exception handling prevents crashes.
6. Topic 6: Functions help organize code.
7. Topic 7: APIs allow programs to communicate.

Total number of lines: 7

Searching for keyword: "python"
Topic 1: Variables store data. Python is dynamically typed.

========== TASK 2 - API INTEGRATION ==========

ID | Title | Category | Price | Rating
----------------------------------------------------------------------
1 | Essence Mascara Lash Princess | beauty | $9.99 | 2.56
2 | Eyeshadow Palette with Mirror | beauty | $19.99 | 2.86
3 | Powder Canister | beauty | $14.99 | 4.64
4 | Red Lipstick | beauty | $12.99 | 4.36
5 | Red Nail Polish | beauty | $8.99 | 4.32
6 | Calvin K